# Data Cleaning: Telco Customer Churn


## 1. Load the data

In [1]:
import pandas as pd

df = pd.read_csv("telco_raw.csv")
raw_rows = len(df)
print("Rows, columns:", df.shape)
df.head()

Rows, columns: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. First look: types, missing values, summary

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [3]:
df.isna().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [4]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


**What to notice:** `TotalCharges` should be a number, but pandas shows it as text (`object`).
That means some cells contain something that is not a number.

## 3. Duplicates

In [5]:
print("Duplicate full rows:      ", df.duplicated().sum())
print("Duplicate customer IDs:    ", df["customerID"].duplicated().sum())

Duplicate full rows:       0
Duplicate customer IDs:     0


## 4. Fix `TotalCharges` (text with blank spaces)

In [6]:
blank = df["TotalCharges"].str.strip() == ""
print("Blank TotalCharges:", blank.sum())
df.loc[blank, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

Blank TotalCharges: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


All of these customers have `tenure = 0`: they just joined and have not been billed yet.
So the blank is **not an error, it really means 0 charged so far**.
Decision: convert to a number and fill with **0**. (The alternative, deleting the 11 rows, would throw away real customers.)

In [7]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].str.strip(), errors="coerce")
print("NaN after conversion:", df["TotalCharges"].isna().sum())
df["TotalCharges"] = df["TotalCharges"].fillna(0)
print("dtype now:", df["TotalCharges"].dtype)

NaN after conversion: 11
dtype now: float64


## 5. Inconsistent categories

In [8]:
cat_cols = df.select_dtypes(exclude="number").columns.drop("customerID")
for c in cat_cols:
    print(c, "->", sorted(df[c].unique()))

gender -> ['Female', 'Male']
Partner -> ['No', 'Yes']
Dependents -> ['No', 'Yes']
PhoneService -> ['No', 'Yes']
MultipleLines -> ['No', 'No phone service', 'Yes']
InternetService -> ['DSL', 'Fiber optic', 'No']
OnlineSecurity -> ['No', 'No internet service', 'Yes']
OnlineBackup -> ['No', 'No internet service', 'Yes']
DeviceProtection -> ['No', 'No internet service', 'Yes']
TechSupport -> ['No', 'No internet service', 'Yes']
StreamingTV -> ['No', 'No internet service', 'Yes']
StreamingMovies -> ['No', 'No internet service', 'Yes']
Contract -> ['Month-to-month', 'One year', 'Two year']
PaperlessBilling -> ['No', 'Yes']
PaymentMethod -> ['Bank transfer (automatic)', 'Credit card (automatic)', 'Electronic check', 'Mailed check']
Churn -> ['No', 'Yes']


**Problem:** several columns have `No internet service` or `No phone service`, which just means `No`.
The information is already in `InternetService` / `PhoneService`, so keeping three values is redundant and confuses models.
Decision: replace both with `No`.

In [9]:
df = df.replace({"No internet service": "No", "No phone service": "No"})

for c in ["MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
          "TechSupport", "StreamingTV", "StreamingMovies"]:
    print(c, "->", sorted(df[c].unique()))

MultipleLines -> ['No', 'Yes']
OnlineSecurity -> ['No', 'Yes']
OnlineBackup -> ['No', 'Yes']
DeviceProtection -> ['No', 'Yes']
TechSupport -> ['No', 'Yes']
StreamingTV -> ['No', 'Yes']
StreamingMovies -> ['No', 'Yes']


## 6. Data types

In [10]:
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

df.dtypes

customerID              str
gender                  str
SeniorCitizen           str
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
Churn                 int64
dtype: object

## 7. Outliers and impossible values

In [11]:
print("Negative values:", (df[["tenure", "MonthlyCharges", "TotalCharges"]] < 0).sum().sum())
print(df[["tenure", "MonthlyCharges", "TotalCharges"]].describe().round(2))

Negative values: 0
        tenure  MonthlyCharges  TotalCharges
count  7043.00         7043.00       7043.00
mean     32.37           64.76       2279.73
std      24.56           30.09       2266.79
min       0.00           18.25          0.00
25%       9.00           35.50        398.55
50%      29.00           70.35       1394.55
75%      55.00           89.85       3786.60
max      72.00          118.75       8684.80


In [12]:
for c in ["tenure", "MonthlyCharges", "TotalCharges"]:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    n = ((df[c] < q1 - 1.5 * iqr) | (df[c] > q3 + 1.5 * iqr)).sum()
    print(f"{c}: {n} outliers")

tenure: 0 outliers
MonthlyCharges: 0 outliers
TotalCharges: 0 outliers


## 8. Clean column names

In [13]:
df.columns = (df.columns.str.strip()
              .str.replace(r"(?<!^)(?=[A-Z])", "_", regex=True)
              .str.lower())
df = df.rename(columns={"customer_i_d": "customer_id", "streaming_t_v": "streaming_tv"})
df.columns.tolist()

['customer_id',
 'gender',
 'senior_citizen',
 'partner',
 'dependents',
 'tenure',
 'phone_service',
 'multiple_lines',
 'internet_service',
 'online_security',
 'online_backup',
 'device_protection',
 'tech_support',
 'streaming_tv',
 'streaming_movies',
 'contract',
 'paperless_billing',
 'payment_method',
 'monthly_charges',
 'total_charges',
 'churn']

## 9. Final checks and save

In [14]:
assert df.duplicated().sum() == 0
assert df.isna().sum().sum() == 0
assert df["customer_id"].is_unique
assert set(df["churn"].unique()) == {0, 1}

print("Rows before:", raw_rows, "| after:", len(df))
print("Churn rate: %.2f%%" % (100 * df["churn"].mean()))
df.to_csv("clean_churn.csv", index=False)
print("Saved clean_churn.csv")

Rows before: 7043 | after: 7043
Churn rate: 26.54%
Saved clean_churn.csv
